# pre-processing anndata object

In [ ]:
import scanpy as sc
import anndata
from scipy import io
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np
import os
import pandas as pd
#import random
#random.seed(1234)

In [ ]:
# load sparse matrix:
X = io.mmread("counts.mtx")

In [ ]:
# create anndata object
adata = anndata.AnnData(
    X=X.transpose().tocsr()
)

In [ ]:
# load cell metadata:
cell_meta = pd.read_csv("metadata.csv")

In [ ]:
# load gene names:
with open("gene_names.csv", 'r') as f:
    gene_names = f.read().splitlines()

In [ ]:
# set anndata observations and index obs by barcodes, var by gene names
adata.obs = cell_meta
adata.obs.index = adata.obs['barcode']
adata.var.index = gene_names


In [ ]:
# load dimensional reduction:
pca = pd.read_csv("pca.csv")
pca.index = adata.obs.index

In [ ]:
# set pca and umap
adata.obsm['X_pca'] = pca.to_numpy()
adata.obsm['X_umap'] = np.vstack((adata.obs['UMAP_1'].to_numpy(), adata.obs['UMAP_2'].to_numpy())).T

In [ ]:
# plot a UMAP colored by seurat clusters to test:
sc.pl.umap(adata, color=['CellType'], frameon=False, save=True)

In [ ]:
# save dataset as anndata format
adata.write('Axin2_combined_sub.h5ad')

# Load data

In [ ]:
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import pandas as pd
import anndata as ad

In [ ]:
scv.settings.verbosity = 3  # show errors(0), warnings(1), info(2), hints(3)
scv.settings.presenter_view = True  # set max width size for presenter view
scv.settings.set_figure_params('scvelo', facecolor='white', dpi=300, frameon=False)
cr.settings.verbosity = 2

In [ ]:
# reload dataset
adata = sc.read_h5ad('Axin2_combined_sub.h5ad')

In [ ]:
# load loom files for spliced/unspliced matrices for each sample:
ldata1 = scv.read('Axin2_1.loom', cache=True)
ldata2 = scv.read('Axin2_2.loom', cache=True)

In [ ]:
# rename barcodes in order to merge:
barcodes = [bc.split(':')[1] for bc in ldata1.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_10' for bc in barcodes]
ldata1.obs.index = barcodes

In [ ]:
barcodes = [bc.split(':')[1] for bc in ldata2.obs.index.tolist()]
barcodes = [bc[0:len(bc)-1] + '_11' for bc in barcodes]
ldata2.obs.index = barcodes

In [ ]:
# make variable names unique
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()

In [ ]:
# concatenate the three loom
ldata = ldata1.concatenate([ldata2])

In [ ]:
# merge matrices into the original adata object
adata = scv.utils.merge(adata, ldata)

In [ ]:
adata

In [ ]:
# plot umap to check
sc.pl.umap(adata, color='CellType', frameon=False, legend_loc='on data', legend_fontsize=8, title='', save='_celltypes.pdf')

In [ ]:
 sc.pl.umap(adata, color=["Gli1","Pdgfra","Cd200","Msx1","Thy1","Ibsp","Sparc","Col1a1",'CellType'], 
               s=50, frameon=False, ncols=2, vmax='p99')

# Computing RNA velocity using scVelo

In [ ]:
scv.pl.proportions(adata, groupby='CellType', save='proportions.pdf')

In [ ]:
# pre-process
scv.pp.filter_and_normalize(adata)
scv.pp.moments(adata)

In [ ]:
# Dynamic model
#scv.tl.recover_dynamics(adata)
#scv.tl.velocity(adata, mode='dynamical')
#scv.tl.velocity_graph(adata)
# Save h5ad for dynamic model data
#adata.write('Axin2_subset_dynamic_velocity.h5ad', compression='gzip')
#adata = scv.read('Axin2_subset_dynamic_velocity.h5ad')

In [ ]:
# compute velocity
#scv.tl.velocity(adata)
scv.tl.velocity(adata, mode='deterministic')
scv.tl.velocity_graph(adata)

# Visualize velocity fields

In [ ]:
scv.pl.velocity_embedding(adata, basis='umap', frameon=False, save='embedding.pdf')


In [ ]:
scv.pl.velocity_embedding_grid(adata, basis='umap', color='CellType', save='embedding_grid.pdf', title='', scale=1.5)


In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap', color=['CellType'], fontsize=5, xlim = [-15,10], ylim = [-8,8], save='embedding_stream.pdf', title='RNA velocity')


In [ ]:
scv.pl.velocity_embedding(adata, arrow_length=8, color=['CellType'], xlim = [-15,10], ylim = [-8,8], arrow_size=3, dpi=300)


In [ ]:
# plot velocity of a selected gene
scv.pl.velocity(adata, 'Gli1',layers=['velocity','X'], perc=None, color='CellType')
scv.pl.velocity(adata, 'Pdgfrb', layers=['velocity','X'], perc=None, color='CellType')
scv.pl.velocity(adata, 'Eng', layers=['velocity','X'], perc=None, color='CellType')
scv.pl.velocity(adata, 'Atxn1', layers=['velocity','X'], perc=None, color='CellType')
scv.pl.velocity(adata, 'Ibsp', layers=['velocity','X'], perc=None, color='CellType')
scv.pl.velocity(adata, 'Col1a2', layers=['velocity','X'], perc=None, color='CellType')


In [ ]:
# Kinetic rate parameters (only run if dynamic model is used)
#df = adata.var
#df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

#kwargs = dict(xscale='log', fontsize=16)
#with scv.GridSpec(ncols=3) as pl:
#    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
#    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
#    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

#scv.get_df(adata, 'fit*', dropna=True).head()


# Downstream analysis

In [ ]:
scv.tl.rank_velocity_genes(adata, groupby='CellType', min_corr=.3)

In [ ]:
df = scv.DataFrame(adata.uns['rank_velocity_genes']['names'])
df.head()

In [ ]:
scv.pl.scatter(adata, df['SuSC'][:6], ylabel='SuSC', ncols=2, frameon=False, color='CellType', size=10, linewidth=1.5)


In [ ]:
# only used with dynamical mode
#top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index
#scv.pl.scatter(adata, basis=top_genes[:15], ncols=5, frameon=False)

In [ ]:
# Cluster-specific top-likelihood genes, only used with dynamical mode
#scv.tl.rank_dynamical_genes(adata, groupby='CellType')
#df = scv.get_df(adata, 'rank_dynamical_genes/names')
#df.head(5)

In [ ]:
for cluster in ['SuSC', 'OP', 'OB-I','OB-II']:
    scv.pl.scatter(adata, df[cluster][:5], color='CellType',ylabel=cluster, frameon=False)


In [ ]:
scv.tl.velocity_confidence(adata)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
df = adata.obs.groupby('CellType')[keys].mean().T
df.style.background_gradient(cmap='coolwarm', axis=1)

In [ ]:
scv.pl.velocity_graph(adata, threshold=.1, color='CellType', legend_fontsize=8)


In [ ]:
x, y = scv.utils.get_cell_transitions(adata, basis='umap', starting_cell=20)
ax = scv.pl.velocity_graph(adata, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata, x=x, y=y, s=70, c='ascending', cmap='gnuplot', ax=ax)

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
# Calculate latent time only when dynamical mode is used
#scv.tl.recover_dynamics(adata)
#scv.tl.latent_time(adata)
#scv.pl.scatter(adata, color='latent_time', color_map='gnuplot')

In [ ]:
# this is needed due to a current bug - bugfix is coming soon.
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']